In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import lingam
import networkx as nx
import matplotlib.pyplot as plt

#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"
#test it works
cp_root

#path to put results
output_dir = project_root/"results"/"graphs_LiM"
output_dir.mkdir(parents=True,exist_ok=True)

In [31]:
"""LiM expects a numeric matrix with continuous and discrete columns as well as
an array per variable indicating 0 for discrete and 1 for continuous.
Similar to what we did for DAGBagM, we write a helper function to infer the datatype."""

def infer_type(df):
    """
    Infers a 'flag array' for a given dataframe.
    0 corresponds to discrete variables, 1 to continuous variables
    """

    flags=[]
    for col in df.columns:
        x = df[col].dropna()
        #classify numeric columns
        if np.issubdtype(x.dtype, np.number):
            vals = x.unique()
            if len(vals) <= 1:
                flags.append(0) #variable is discrete
            else:
                flags.append(1) # variable is continuous
    return np.asarray(flags, dtype=int)

def run_lim(df, seed=1):
    """
    Clean dataframe and generate the numeric matrix as well as flag array.
    Run LiM on the data.
    """

    df_clean=df.dropna().copy()
    flags = infer_type(df_clean)
    flags_2d = flags.reshape(1,-1)
    cols = list(df_clean.columns)

    X = df_clean.to_numpy(dtype=float)

    model = lingam.LiM()

    model.fit(X, flags_2d, only_global=True)
    #LiM creates an adjacency matrix
    M = model.adjacency_matrix_
    #coerce adjacency matrix into a matrix of 0's and 1's
    adj = (np.abs(M) > 0).astype(int)

    return adj, cols

In [8]:
def draw_graph(adj, nodes, output_path):
    G = nx.DiGraph()
    #add nodes
    G.add_nodes_from(nodes)
    
    #add directed edges, adj[i,j] = 1 => i -> j
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                G.add_edge(src, tgt)

    plt.figure(figsize=(10,8))
    pos = nx.spring_layout(G, k=1.2, iterations=500, seed=0)
    nx.draw(G, pos, with_labels=True, labels={node: node for node in nodes},
            node_size=900, font_size=8, arrowsize=10)
    plt.savefig(out_path, dpi=150)
    plt.close()

In [11]:
def one_hot_encode(df):
    non_numeric_cols = df.select_dtypes(exclude=["number"]).columns

    #return the same df if no non numeric
    if len(non_numeric_cols) == 0:
        return df.copy()

    df_encoded = pd.get_dummies(
        df,
        columns=list(non_numeric_cols),
        drop_first=False,   # keep full one-hot encoding [web:195][web:198]
        dtype=int,
    )
    return df_encoded

In [ ]:
csv_files = sorted(cp_root.rglob("*.csv"))

for csv_path in csv_files:
    #relative path
    rel = csv_path.relative_to(cp_root)
    print(f"Processing: {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue
        df_enc = one_hot_encode(df)
        # Run LiM on this dataset
        adj,nodes = run_lim(df_enc, seed=1)

        #Output schema scenario__file__LiM.png
        parts = rel.parts           
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__LiM.png"
        out_path = output_dir / out_name

        # Save PNG
        draw_graph(adj, nodes, out_path)
        print(f"  Saved graph to {out_path.relative_to(project_root)}")

    except Exception as e:
        print(f"  ERROR on {rel}: {e}")

Processing: berkson_paradox/.ipynb_checkpoints/admission_bias-checkpoint.csv
W_est (without the 2nd phase) is: 
 [[0.        0.       ]
 [0.3279654 0.       ]]
  Saved graph to results/graphs_LiM/berkson_paradox__admission_bias-checkpoint__LiM.png
Processing: berkson_paradox/.ipynb_checkpoints/loan_approval_bias-checkpoint.csv
W_est (without the 2nd phase) is: 
 [[0.         0.51445447]
 [0.         0.        ]]
  Saved graph to results/graphs_LiM/berkson_paradox__loan_approval_bias-checkpoint__LiM.png
Processing: berkson_paradox/.ipynb_checkpoints/movie_success_bias-checkpoint.csv
W_est (without the 2nd phase) is: 
 [[0.         0.16718244]
 [0.         0.        ]]
  Saved graph to results/graphs_LiM/berkson_paradox__movie_success_bias-checkpoint__LiM.png
Processing: berkson_paradox/admission_bias.csv
W_est (without the 2nd phase) is: 
 [[0.        0.       ]
 [0.3279654 0.       ]]
  Saved graph to results/graphs_LiM/berkson_paradox__admission_bias__LiM.png
Processing: berkson_parad

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:263: RuntimeWarning: overflow encountered in scalar multiply
  obj = loss + 0.5 * rho * h * h + alpha * h + self._lambda1 * w.sum()
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:265: RuntimeWarning: overflow encountered in multiply
  G_smooth = G_loss + (rho * h + alpha) * G_h.T * W * 2  # 2019
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:265: RuntimeWarning: invalid value encountered in multiply
  G_smooth = G_loss + (rho * h + alpha) * G_h.T * W * 2  # 2019
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:117: RuntimeWarning: invalid value encountered in logaddexp
  (np.logaddexp(0, M) - X * M) * np.absolute(dis_con - 1)


W_est (without the 2nd phase) is: 
 [[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 3.46322201e+02  0.00000000e+00  0.00000000e+00  0.00000000e+00
   1.25776986e+02  1.17563564e-01  0.00000000e+00]
 [ 1.73281870e+02  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 3.21965437e+02  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 1.77074129e+02  0.00000000e+00  0.00000000e+00 -1.39519611e+01
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 3.22395787e+02  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  1.70790005e+01  0.00000000e+00
   0.00000000e+00  1.90914499e-01  0.00000000e+00]]
  Saved graph to results/graphs_LiM/casual_effect__device_failure_data__LiM.png
Processing: casual_effect/machine_maintenance_data.csv
W_est (wi

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:263: RuntimeWarning: overflow encountered in scalar multiply
  obj = loss + 0.5 * rho * h * h + alpha * h + self._lambda1 * w.sum()
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:265: RuntimeWarning: overflow encountered in multiply
  G_smooth = G_loss + (rho * h + alpha) * G_h.T * W * 2  # 2019
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:265: RuntimeWarning: invalid value encountered in multiply
  G_smooth = G_loss + (rho * h + alpha) * G_h.T * W * 2  # 2019
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:117: RuntimeWarning: invalid value encountered in logaddexp
  (np.logaddexp(0, M) - X * M) * np.absolute(dis_con - 1)
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:

W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.61135013  0.        ]
 [-0.22690006  0.          0.          0.67157433]
 [ 0.          0.          0.          0.        ]
 [-0.46177378  0.         -0.49704811  0.        ]]
  Saved graph to results/graphs_LiM/causal_direction_iv__ecommerce_sem__LiM.png
Processing: causal_direction_iv/environment_sem.csv
W_est (without the 2nd phase) is: 
 [[0.         0.85034651 0.         0.22586096]
 [0.         0.         0.         1.1994756 ]
 [1.99865764 0.17662346 0.         0.34979903]
 [0.         0.         0.         0.        ]]
  Saved graph to results/graphs_LiM/causal_direction_iv__environment_sem__LiM.png
Processing: causal_direction_iv/marketing_sem.csv
W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          0.        ]
 [ 0.          0.         -2.15567166  2.07088997]
 [ 1.12472872  0.          0.          0.        ]
 [ 0.          0.          1.66330243  0.        ]]
  Saved graph to results/grap

...

W_est (without the 2nd phase) is: 
 [[ 0.          0.         -5.67417455]
 [ 0.          0.          1.09474223]
 [ 0.          0.          0.        ]]
  Saved graph to results/graphs_LiM/moderation_effect__moderation_effect_sem-checkpoint__LiM.png
Processing: moderation_effect/arthritis_pain_reduction.csv
W_est (without the 2nd phase) is: 
 [[  0.          13.69372177 -14.78034318]
 [  0.           0.          -1.68297629]
 [  0.           0.           0.        ]]
  Saved graph to results/graphs_LiM/moderation_effect__arthritis_pain_reduction__LiM.png
Processing: moderation_effect/depression_symptom_reduction.csv
W_est (without the 2nd phase) is: 
 [[ 0.          0.         -8.0998155 ]
 [ 0.          0.          0.        ]
 [ 0.         -0.14783683  0.        ]]
  Saved graph to results/graphs_LiM/moderation_effect__depression_symptom_reduction__LiM.png
Processing: moderation_effect/hypertension_bp_reduction.csv


/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[  0.           2.38644751 -29.94185379]
 [  0.           0.           0.        ]
 [  0.          -0.33867595   0.        ]]
  Saved graph to results/graphs_LiM/moderation_effect__hypertension_bp_reduction__LiM.png
Processing: moderation_effect/infection_bacteria_reduction.csv
W_est (without the 2nd phase) is: 
 [[ 0.          0.         -1.09824376]
 [ 0.57450148  0.         -4.8814085 ]
 [ 0.          0.          0.        ]]
  Saved graph to results/graphs_LiM/moderation_effect__infection_bacteria_reduction__LiM.png
Processing: moderation_effect/moderation_effect_sem.csv


/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[ 0.          0.         -5.67408709]
 [ 0.          0.          1.09381792]
 [ 0.          0.          0.        ]]
  Saved graph to results/graphs_LiM/moderation_effect__moderation_effect_sem__LiM.png
Processing: necessity_sufficiency/bridge_integrity.csv
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.        ]
 [0.         0.         0.         0.87772071]
 [0.         0.         0.         0.        ]
 [0.42742411 0.         0.40709776 0.        ]]
  Saved graph to results/graphs_LiM/necessity_sufficiency__bridge_integrity__LiM.png
Processing: necessity_sufficiency/factory_monitoring.csv
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.83632765]
 [0.42264657 0.44368063 0.         0.        ]]
  Saved graph to results/graphs_LiM/necessity_sufficiency__factory_monitoring__LiM.png
Processing: necessity_suff